# Deep Retention — Preprocesamiento de Datos

Implementación académica y reproducible del preprocesamiento para la futura red neuronal MLP.

Este notebook limpia una copia del dataset, selecciona las nueve variables autorizadas, crea particiones estratificadas y ajusta un `ColumnTransformer` exclusivamente con entrenamiento. No modifica el CSV, no exporta artefactos y no entrena modelos.

## 1. Carga y limpieza en memoria

El dataset se localiza con rutas relativas válidas desde la raíz del proyecto o desde `notebooks/`. Se reutilizan únicamente los criterios de limpieza definidos en el EDA, sin repetir el análisis exploratorio.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_FILENAME = "Deep_Retention_Dataset.csv"
DATA_CANDIDATES = [
    Path("data") / DATA_FILENAME,
    Path("..") / "data" / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"No se encontró {DATA_FILENAME} desde la raíz ni desde notebooks/.")

FEATURES = [
    "CreditScore", "Geography", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
TARGET = "Exited"
NUMERIC_FEATURES = [
    "CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary",
]
CATEGORICAL_FEATURES = ["Geography"]
BINARY_FEATURES = ["HasCrCard", "IsActiveMember"]
EXCLUDED_FEATURES = ["CustomerId", "Surname", "Gender", TARGET]

print(f"Archivo: {DATA_PATH.as_posix()}")
print(f"Scikit-learn: {sklearn.__version__}")

Archivo: ../data/Deep_Retention_Dataset.csv
Scikit-learn: 1.9.1


In [2]:
df_original = pd.read_csv(DATA_PATH)
df_clean = df_original.copy()

text_columns = df_clean.select_dtypes(include="object").columns
for column in text_columns:
    df_clean[column] = df_clean[column].str.strip()

corrupted_surnames = df_clean["Surname"].str.contains("?", regex=False, na=False)
corrupted_surname_count = int(corrupted_surnames.sum())
df_clean.loc[corrupted_surnames, "Surname"] = pd.NA

text_spaces_after = 0
for column in text_columns:
    values = df_clean[column].dropna().astype(str)
    text_spaces_after += int((values != values.str.strip()).sum())

cleaning_summary = pd.DataFrame({
    "control": [
        "Filas originales", "Filas después de limpiar", "Apellidos con '?' detectados",
        "Apellidos con '?' restantes", "Valores faltantes originales",
        "Valores faltantes después de limpiar", "Faltantes en Surname",
        "Faltantes predictivos", "Textos con espacios exteriores",
        "CustomerId modificados",
    ],
    "resultado": [
        len(df_original), len(df_clean), corrupted_surname_count,
        int(df_clean["Surname"].str.contains("?", regex=False, na=False).sum()),
        int(df_original.isna().sum().sum()), int(df_clean.isna().sum().sum()),
        int(df_clean["Surname"].isna().sum()), int(df_clean[FEATURES].isna().sum().sum()),
        text_spaces_after, int((df_clean["CustomerId"] != df_original["CustomerId"]).sum()),
    ],
})
display(cleaning_summary)

,control,resultado
0,Filas originales,10000
1,Filas después de limpiar,10000
2,Apellidos con '?' detectados,92
3,Apellidos con '?' restantes,0
4,Valores faltantes originales,0
5,Valores faltantes después de limpiar,92
6,Faltantes en Surname,92
7,Faltantes predictivos,0
8,Textos con espacios exteriores,0
9,CustomerId modificados,0


**Resultado.** La limpieza se aplica exclusivamente a `df_clean`, una copia en memoria. Conserva los 10.000 clientes y todos sus identificadores, elimina espacios exteriores y reemplaza los 92 apellidos con `?` por `pd.NA`. Como `Surname` está excluido de `X`, los 92 faltantes resultantes no afectan las nueve características predictoras.

## 2. Selección de variables

`X` contiene exclusivamente las nueve predictoras aprobadas y `y` conserva la etiqueta histórica `Exited`. Los identificadores, `Surname`, `Gender` y todas las columnas financieras simuladas quedan fuera de la matriz de entrada, pero permanecen intactos en el DataFrame.

In [3]:
X = df_clean[FEATURES].copy()
y = df_clean[TARGET].copy()

financial_columns = [column for column in df_clean.columns if column.endswith("_Sim")]
excluded_columns = EXCLUDED_FEATURES + financial_columns
excluded_columns_in_X = sorted(set(X.columns) & set(excluded_columns))

if X.columns.tolist() != FEATURES:
    raise ValueError("Las columnas de X no coinciden con las nueve variables autorizadas.")
if X.shape[1] != 9:
    raise ValueError("X debe contener exactamente nueve variables.")
if excluded_columns_in_X:
    raise ValueError(f"X contiene columnas excluidas: {excluded_columns_in_X}")

selection_summary = pd.DataFrame({
    "objeto": ["X", "y"],
    "filas": [X.shape[0], y.shape[0]],
    "columnas": [X.shape[1], 1],
    "faltantes": [int(X.isna().sum().sum()), int(y.isna().sum())],
})
display(selection_summary)
print("Variables de X:", X.columns.tolist())
print("Columnas excluidas presentes en X:", excluded_columns_in_X)

,objeto,filas,columnas,faltantes
0,X,10000,9,0
1,y,10000,1,0


Variables de X: ['CreditScore', 'Geography', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Columnas excluidas presentes en X: []


## 3. División de entrenamiento, validación y prueba

La división se realiza antes de ajustar transformaciones. Primero se reserva el 30 % y luego se divide esa parte por la mitad, obteniendo 70 % para entrenamiento, 15 % para validación y 15 % para prueba. Ambas operaciones usan `stratify` y `random_state=42`.

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

In [5]:
split_summary = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Validación", "Prueba"],
    "registros": [len(X_train), len(X_val), len(X_test)],
    "porcentaje_dataset": [len(X_train) / len(X) * 100, len(X_val) / len(X) * 100, len(X_test) / len(X) * 100],
    "abandonos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "porcentaje_abandono": [y_train.mean() * 100, y_val.mean() * 100, y_test.mean() * 100],
})
display(split_summary)

train_indices = set(X_train.index)
val_indices = set(X_val.index)
test_indices = set(X_test.index)

split_checks = pd.DataFrame({
    "comprobación": [
        "Entrenamiento ∩ validación", "Entrenamiento ∩ prueba",
        "Validación ∩ prueba", "Suma de tamaños",
    ],
    "resultado": [
        len(train_indices & val_indices), len(train_indices & test_indices),
        len(val_indices & test_indices), len(X_train) + len(X_val) + len(X_test),
    ],
    "esperado": [0, 0, 0, len(X)],
})
display(split_checks)

,conjunto,registros,porcentaje_dataset,abandonos,porcentaje_abandono
0,Entrenamiento,7000,70.0,1426,20.371429
1,Validación,1500,15.0,306,20.400000
2,Prueba,1500,15.0,305,20.333333


,comprobación,resultado,esperado
0,Entrenamiento ∩ validación,0,0
1,Entrenamiento ∩ prueba,0,0
2,Validación ∩ prueba,0,0
3,Suma de tamaños,10000,10000


**Interpretación.** Los tres conjuntos no comparten índices y conservan aproximadamente la tasa histórica de abandono. El conjunto de prueba queda aislado: no se usa para ajustar transformadores ni para tomar decisiones de diseño.

## 4. Clasificación de las variables

- **Numéricas para escalado:** `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts` y `EstimatedSalary`.
- **Categórica nominal:** `Geography`.
- **Binarias:** `HasCrCard` e `IsActiveMember`.

Las variables binarias ya utilizan `0` y `1`; se conservan sin escalado ni codificación adicional.

In [6]:
feature_groups = pd.DataFrame({
    "grupo": ["Numéricas", "Categórica nominal", "Binarias"],
    "cantidad": [len(NUMERIC_FEATURES), len(CATEGORICAL_FEATURES), len(BINARY_FEATURES)],
    "variables": [
        ", ".join(NUMERIC_FEATURES),
        ", ".join(CATEGORICAL_FEATURES),
        ", ".join(BINARY_FEATURES),
    ],
})
display(feature_groups)

,grupo,cantidad,variables
0,Numéricas,6,"CreditScore, Age, Tenure, Balance, NumOfProduc..."
1,Categórica nominal,1,Geography
2,Binarias,2,"HasCrCard, IsActiveMember"


## 5. Escalado, codificación y `ColumnTransformer`

`StandardScaler` estandariza cada variable numérica mediante:

\[
z = \frac{x - \text{media del entrenamiento}}{\text{desviación del entrenamiento}}
\]

Esto evita que magnitudes como `Balance` o `EstimatedSalary` dominen a variables de menor escala. `OneHotEncoder` representa `Geography` sin imponer un orden artificial y usa `handle_unknown="ignore"` para tolerar categorías nuevas durante la inferencia. No se incluye imputación porque las predictoras no contienen faltantes.

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        (
            "geography",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
        ("binary", "passthrough", BINARY_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(preprocessor)

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['CreditScore', 'Age', 'Tenure', 'Balance',
                                  'NumOfProducts', 'EstimatedSalary']),
                                ('geography',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Geography']),
                                ('binary', 'passthrough',
                                 ['HasCrCard', 'IsActiveMember'])],
                  verbose_feature_names_out=False)


### Aplicación sin fuga de información

El preprocesador se ajusta una sola vez con `X_train`. Validación y prueba se transforman después con los parámetros aprendidos en entrenamiento; nunca reciben `fit` ni `fit_transform`.

In [8]:
X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_processed = preprocessor.transform(X_val).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)

feature_names = preprocessor.get_feature_names_out()
y_train_array = y_train.to_numpy(dtype=np.float32)
y_val_array = y_val.to_numpy(dtype=np.float32)
y_test_array = y_test.to_numpy(dtype=np.float32)

transformed_shapes = pd.DataFrame({
    "conjunto": ["Entrenamiento", "Validación", "Prueba"],
    "forma_original": [X_train.shape, X_val.shape, X_test.shape],
    "forma_transformada": [X_train_processed.shape, X_val_processed.shape, X_test_processed.shape],
    "tipo": [X_train_processed.dtype, X_val_processed.dtype, X_test_processed.dtype],
})
display(transformed_shapes)
print("Características finales:", len(feature_names))
print(feature_names.tolist())

,conjunto,forma_original,forma_transformada,tipo
0,Entrenamiento,"(7000, 9)","(7000, 11)",float32
1,Validación,"(1500, 9)","(1500, 11)",float32
2,Prueba,"(1500, 9)","(1500, 11)",float32


Características finales: 11
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'HasCrCard', 'IsActiveMember']


## 6. Verificación de resultados

Se comprueban dimensiones, valores finitos, categorías aprendidas, conservación de las variables binarias y ajuste del escalador únicamente con entrenamiento.

In [9]:
scaler = preprocessor.named_transformers_["numeric"]
encoder = preprocessor.named_transformers_["geography"]

has_card_index = feature_names.tolist().index("HasCrCard")
active_member_index = feature_names.tolist().index("IsActiveMember")
geography_indices = [
    feature_names.tolist().index(name)
    for name in encoder.get_feature_names_out(CATEGORICAL_FEATURES)
]

unknown_geography_example = X_train.iloc[[0]].copy()
unknown_geography_example["Geography"] = "Unknown"
unknown_geography_processed = preprocessor.transform(unknown_geography_example).astype(np.float32)

verification_results = pd.DataFrame({
    "comprobación": [
        "Mismo número de características", "Entrenamiento sin NaN o infinitos",
        "Validación sin NaN o infinitos", "Prueba sin NaN o infinitos",
        "HasCrCard conservada", "IsActiveMember conservada",
        "Scaler ajustado con 7.000 registros", "Medias del scaler corresponden a X_train",
        "Geography desconocida se ignora", "Particiones sin registros compartidos",
    ],
    "resultado": [
        X_train_processed.shape[1] == X_val_processed.shape[1] == X_test_processed.shape[1],
        bool(np.isfinite(X_train_processed).all()),
        bool(np.isfinite(X_val_processed).all()),
        bool(np.isfinite(X_test_processed).all()),
        bool(np.array_equal(X_train_processed[:, has_card_index], X_train["HasCrCard"].to_numpy(np.float32))),
        bool(np.array_equal(X_train_processed[:, active_member_index], X_train["IsActiveMember"].to_numpy(np.float32))),
        int(scaler.n_samples_seen_) == len(X_train),
        bool(np.allclose(scaler.mean_, X_train[NUMERIC_FEATURES].mean().to_numpy())),
        bool(np.allclose(unknown_geography_processed[0, geography_indices], 0.0)),
        not (train_indices & val_indices or train_indices & test_indices or val_indices & test_indices),
    ],
})
display(verification_results)

print("Categorías aprendidas para Geography:", encoder.categories_[0].tolist())
print("Columnas one-hot:", encoder.get_feature_names_out(CATEGORICAL_FEATURES).tolist())

,comprobación,resultado
0,Mismo número de características,True
1,Entrenamiento sin NaN o infinitos,True
2,Validación sin NaN o infinitos,True
3,Prueba sin NaN o infinitos,True
4,HasCrCard conservada,True
5,IsActiveMember conservada,True
6,Scaler ajustado con 7.000 registros,True
7,Medias del scaler corresponden a X_train,True
8,Geography desconocida se ignora,True
9,Particiones sin registros compartidos,True


Categorías aprendidas para Geography: ['France', 'Germany', 'Spain']
Columnas one-hot: ['Geography_France', 'Geography_Germany', 'Geography_Spain']


In [10]:
scaler_parameters = pd.DataFrame({
    "variable": NUMERIC_FEATURES,
    "media_entrenamiento": scaler.mean_,
    "desviación_entrenamiento": scaler.scale_,
})
display(scaler_parameters)

transformed_example = pd.DataFrame(
    X_train_processed[:5],
    columns=feature_names,
    index=X_train.index[:5],
)
display(transformed_example)

,variable,media_entrenamiento,desviación_entrenamiento
0,CreditScore,650.562429,96.532561
1,Age,38.998429,10.574194
2,Tenure,5.010571,2.891693
3,Balance,76314.462261,62370.118628
4,NumOfProducts,1.533000,0.584120
5,EstimatedSalary,99466.316050,57447.307427


,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,HasCrCard,IsActiveMember
9136,-0.575582,-1.796679,-0.349474,0.312906,-0.912483,1.362512,1.0,0.0,0.0,1.0,0.0
6410,0.398182,2.269825,-1.386929,-1.223574,0.799493,1.515472,0.0,0.0,1.0,1.0,1.0
2607,1.693082,-0.756410,-1.386929,0.682321,-0.912483,0.038955,1.0,0.0,0.0,1.0,0.0
3201,0.740036,-1.796679,0.342162,-1.223574,0.799493,1.671444,1.0,0.0,0.0,1.0,0.0
3330,0.905783,-1.985818,-0.349474,-1.223574,0.799493,-0.899384,1.0,0.0,0.0,1.0,1.0


**Resultado.** Las seis variables numéricas quedan estandarizadas con parámetros aprendidos en entrenamiento. `Geography` produce tres columnas one-hot para las categorías observadas (`France`, `Germany` y `Spain`) y las dos variables binarias conservan sus valores. El resultado contiene 11 características numéricas de tipo `float32`, sin valores `NaN` ni infinitos.

## 7. Preparación para el entrenamiento posterior

Quedan disponibles:

- `X_train_processed` y `y_train_array`: entrenamiento del MLP.
- `X_val_processed` y `y_val_array`: validación, selección de arquitectura y control de sobreajuste.
- `X_test_processed` y `y_test_array`: evaluación final después de elegir el modelo.

La semilla `42` y el procedimiento de dos divisiones permiten reproducir las particiones. Durante la inferencia en Streamlit deberá reutilizarse exactamente el mismo `preprocessor` ajustado; no se reconstruirán manualmente el escalado ni la codificación.

En esta fase no se exporta el preprocesador, no se crean CSV transformados y no se entrena ni guarda una red neuronal. La serialización con Joblib se realizará junto con el entrenamiento, usando exactamente el transformador asociado al modelo aprobado.